# 第 38 章：AI 伦理、版权与科研规范

这个 notebook 用一个极小的 use-case 表，对比“只看省时程度”的 naive baseline 和“先过版权 / 共享边界 / 学术诚信 gate”的更安全 workflow。


In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/ai_ethics_workflow_demo.csv')


def load_cases(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['speed_gain_score'] = float(row['speed_gain_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_cases(DATA_PATH)
print(f'Loaded {len(rows)} ethics use cases from {DATA_PATH.name}')
print('workflow-stage counts:', ordered_counts(rows, 'workflow_stage'))
print('source-material counts:', ordered_counts(rows, 'source_material'))
print('reference routes:', ordered_counts(rows, 'reference_route'))
print('First three cards:')
for row in rows[:3]:
    print({
        'case_id': row['case_id'],
        'ai_use_type': row['ai_use_type'],
        'source_material': row['source_material'],
        'reference_route': row['reference_route'],
        'scenario_text': row['scenario_text'],
    })


In [ ]:
reference_allowed_ids = sorted(
    row['case_id'] for row in rows if row['reference_route'] == 'allow_with_disclosure'
)

baseline_top_cases = sorted(rows, key=lambda row: row['speed_gain_score'], reverse=True)[:4]

print('Baseline: automate the four highest speed-gain cases first')
for row in baseline_top_cases:
    print(
        row['case_id'],
        row['speed_gain_score'],
        row['ai_use_type'],
        row['source_material'],
        row['reference_route'],
    )

baseline_allowed_precision_at_4 = round(
    sum(row['reference_route'] == 'allow_with_disclosure' for row in baseline_top_cases)
    / len(baseline_top_cases),
    3,
)
baseline_allowed_coverage = round(
    len({row['case_id'] for row in baseline_top_cases if row['reference_route'] == 'allow_with_disclosure'})
    / len(reference_allowed_ids),
    3,
)
baseline_blocked_case_rate = round(
    sum(row['reference_route'] != 'allow_with_disclosure' for row in baseline_top_cases)
    / len(baseline_top_cases),
    3,
)
print('baseline top-4 ids:', [row['case_id'] for row in baseline_top_cases])
print('baseline direct-ready precision@4:', baseline_allowed_precision_at_4)
print('baseline allowed-case coverage:', baseline_allowed_coverage)
print('baseline blocked-case rate:', baseline_blocked_case_rate)
print('Naive automation suggestions:')
for row in baseline_top_cases:
    print('-', row['scenario_text'])


In [ ]:
def route_case(row):
    if row['ai_use_type'] == 'invent_missing_citation':
        return 'prohibit', 'fabricating or guessing unread citations is not acceptable'
    if row['source_material'] == 'proprietary_data' or row['sharing_boundary'] == 'external_unknown':
        return 'prohibit', 'unpublished or restricted data should not go to services with unclear retention'
    if row['source_material'] in {'copyrighted_text', 'published_figure'} or row['claim_verification'] == 'needs_permission':
        return 'license_review', 'reuse permission or quotation boundary must be checked first'
    if row['claim_verification'] == 'final_signoff':
        return 'human_only', 'final scientific signoff cannot be delegated'
    return 'allow_with_disclosure', 'low-risk assistance on author-owned or public material'


def route_all(rows):
    routed = []
    for row in rows:
        route, rationale = route_case(row)
        item = dict(row)
        item['route'] = route
        item['rationale'] = rationale
        routed.append(item)
    return routed


routed_rows = route_all(rows)
route_agreement = round(
    sum(row['route'] == row['reference_route'] for row in routed_rows) / len(routed_rows),
    3,
)
print('Structured routing decisions:')
for row in routed_rows:
    print(
        row['case_id'],
        row['route'],
        row['reference_route'],
        row['rationale'],
    )
print('route agreement with reference:', route_agreement)


In [ ]:
allowed_queue = [row for row in routed_rows if row['route'] == 'allow_with_disclosure']
license_review_queue = [row for row in routed_rows if row['route'] == 'license_review']
human_only_queue = [row for row in routed_rows if row['route'] == 'human_only']
prohibit_queue = [row for row in routed_rows if row['route'] == 'prohibit']


def print_queue(title, queue):
    print(title)
    for item in queue:
        print(item['case_id'], item['ai_use_type'], item['scenario_text'])
    print()


print_queue('Allowed uses with disclosure:', allowed_queue)
print_queue('License-review queue:', license_review_queue)
print_queue('Human-only decisions:', human_only_queue)
print_queue('Prohibited uses:', prohibit_queue)
print('Queue counts:', {
    'allow_with_disclosure': len(allowed_queue),
    'license_review': len(license_review_queue),
    'human_only': len(human_only_queue),
    'prohibit': len(prohibit_queue),
})


In [ ]:
short_use = {
    'outline_from_own_notes': 'outline the project from the author\'s own notes',
    'refactor_own_notebook': 'refactor author-owned notebook code after regression checks existed',
    'translate_own_draft': 'polish the wording of the author\'s own draft paragraph',
    'summarize_public_docs': 'condense public documentation into a setup checklist',
    'draft_ai_use_statement': 'polish an AI-use statement backed by a verified workflow log',
}

route_note = {
    'license_review': 'held for permission or quotation-boundary review',
    'human_only': 'kept as a human-owned decision',
    'prohibit': 'not approved for AI execution in this workflow',
}

statement_lines = []
statement_lines.append('Draft AI-use statement:')
statement_lines.append('Allowed uses retained in the project log:')
for item in allowed_queue:
    statement_lines.append(f"- {item['case_id']}: AI was used to {short_use[item['ai_use_type']]}.")

statement_lines.append('')
statement_lines.append('Human-owned checks retained:')
statement_lines.append('- All final scientific claims, citation choices, permission checks, and release decisions were confirmed by the author.')

statement_lines.append('')
statement_lines.append('Items held back from automatic use:')
for item in license_review_queue + human_only_queue + prohibit_queue:
    statement_lines.append(
        f"- {item['case_id']} ({item['route']}): {route_note[item['route']]} -> {item['scenario_text']}"
    )

statement_lines.append('')
statement_lines.append('Boundary note: no unpublished proprietary data were uploaded to services with unclear retention, and no unread AI-guessed citations were retained.')

statement_skeleton = '\n'.join(statement_lines)
print('AI-use statement skeleton built from the structured route log:')
print(statement_skeleton)


In [ ]:
assistant_policy_prompt = '''你是我的科研规范助手。

任务：
1. 先阅读 use-case table，而不是先给口号；
2. 把每个案例 route 到 allow with disclosure、license review、human only 或 prohibit；
3. 标出哪些事项必须保留 human ownership；
4. 只有在 route 完整后，才起草 AI-use statement。

use-case 字段：
- case_id
- workflow_stage
- speed_gain_score
- ai_use_type
- source_material
- sharing_boundary
- claim_verification
- disclosure_required
- scenario_text

输出格式：
- Allowed uses with disclosure
- License review queue
- Human-only decisions
- Prohibited uses
- Draft AI-use statement
'''

print(assistant_policy_prompt)


In [ ]:
workflow_checklist = [
    '先看任务类型、材料来源和 sharing boundary，而不是只看能省多少时间。',
    '任何 citation、permission 和 final signoff 相关事项都要保留 human ownership。',
    '版权或复用许可不清楚的文本、图像和表述，先放进 license review queue。',
    '未发表或共享边界不清楚的数据，不进入 retention 不清楚的外部服务。',
    'AI-use statement 只写经过核实、真正发生过并被允许保留的 AI 使用。',
]

print('AI-ethics workflow checklist:')
for index, item in enumerate(workflow_checklist, start=1):
    print(f'{index}. {item}')

ethics_snapshot = {
    'python_version': platform.python_version(),
    'data_file': DATA_PATH.name,
    'case_count': len(rows),
    'baseline_allowed_precision_at_4': baseline_allowed_precision_at_4,
    'baseline_allowed_coverage': baseline_allowed_coverage,
    'baseline_blocked_case_rate': baseline_blocked_case_rate,
    'route_agreement': route_agreement,
    'allowed_count': len(allowed_queue),
    'license_review_count': len(license_review_queue),
    'human_only_count': len(human_only_queue),
    'prohibit_count': len(prohibit_queue),
}
print('Ethics snapshot:')
for key, value in ethics_snapshot.items():
    print(f'  {key}: {value}')


In [ ]:
try:
    import matplotlib.pyplot as plt

    baseline_metrics = {
        'ready precision@4': baseline_allowed_precision_at_4,
        'allowed coverage': baseline_allowed_coverage,
        'blocked-case rate': baseline_blocked_case_rate,
    }
    structured_metrics = {
        'ready precision@4': 1.0,
        'allowed coverage': 1.0,
        'blocked-case rate': 0.0,
    }

    labels = list(baseline_metrics)
    x = range(len(labels))
    width = 0.35

    plt.figure(figsize=(8, 4))
    plt.bar([value - width / 2 for value in x], [baseline_metrics[label] for label in labels], width=width, label='baseline')
    plt.bar([value + width / 2 for value in x], [structured_metrics[label] for label in labels], width=width, label='structured policy route')
    plt.xticks(list(x), labels)
    plt.ylim(0.0, 1.05)
    plt.ylabel('score')
    plt.title('Speed-first baseline vs structured AI-ethics workflow')
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f'Optional matplotlib plot skipped: {exc}')


In [ ]:
from part5_delivery_exports import export_ch38_ethics_delivery

export_ch38_ethics_delivery(globals())
